# lab-religious-protocols compute runner
Managed from GitHub. Do not put research logic in this notebook.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import shutil
import subprocess

REMOTE = 'https://github.com/tndd/lab-religious-protocols.git'
REPO = Path('/tmp/lab-religious-protocols')
OUT = Path('/kaggle/working/results')

shutil.rmtree(REPO, ignore_errors=True)
shutil.rmtree(OUT, ignore_errors=True)
subprocess.run(['git', 'clone', REMOTE, str(REPO)], check=True)

request = json.loads((REPO / 'kaggle' / 'RUN_REQUEST').read_text())
target_ref = request['target_ref']
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', target_ref], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', target_ref], check=True)

job = json.loads((REPO / 'kaggle' / 'job.json').read_text())
print('Target ref:', target_ref)
print('Job:', job['name'])
print('Command:', job['command'])

subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPO)], check=True)
subprocess.run(job['command'], cwd=REPO, shell=True, check=True)

source_results = REPO / job.get('results_dir', 'results')
if not source_results.exists():
    raise FileNotFoundError(f'Expected results directory not found: {source_results}')
shutil.copytree(source_results, OUT)

metadata = {
    'target_ref': target_ref,
    'job': job['name'],
    'command': job['command'],
    'completed_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}
(OUT / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('Results copied to', OUT)
